In [1]:
# Cell 1: Check everything is working
import sys
import os

print("Python version:", sys.version)
print("Current directory:", os.getcwd())

# Check if NVIDIA GPU is available
import subprocess
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print("\nGPU Info:")
    print(result.stdout[:500])  # First 500 chars
except:
    print("nvidia-smi not found - but we'll check via PyTorch below")

Python version: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
Current directory: C:\Users\Disha Saini\Documents\ML\Untitled Folder

GPU Info:
Tue Apr 21 18:47:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.59                 Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


In [3]:
# Cell 2: Verify PyTorch and CUDA
import torch
import torchaudio


print("PyTorch version:", torch.__version__)
print("TorchAudio version:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Make sure CUDA drivers are installed.")



PyTorch version: 2.1.0+cu118
TorchAudio version: 2.1.0+cu118
CUDA available: True
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory: 6.44 GB


In [4]:
# Cell 3: Install everything needed for the ENTIRE project
# Run this once - it will take 5-10 minutes

import subprocess
import sys

def install(package):
    """Helper to install packages and show output"""
    print(f"Installing {package}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {package} installed successfully")
    else:
        print(f"  ✗ ERROR installing {package}:")
        print(result.stderr[-500:])  # Last 500 chars of error
    return result.returncode == 0

# Core packages
packages = [
    "torch==2.1.0",
    "torchaudio==2.1.0", 
    "torchvision==0.16.0",  # sometimes needed
    "numpy==1.24.4",
    "scipy==1.11.4",
    "librosa==0.10.1",
    "soundfile==0.12.1",
    "musdb==0.4.2",        # MUSDB18 loader
    "stempeg==0.2.3",       # For reading MUSDB stems
    "matplotlib==3.7.3",    # For plotting spectrograms
    "tqdm==4.66.1",         # Progress bars
    "ipywidgets==8.1.1",    # Jupyter widgets
    "jupyter-lab==4.0.8",   # Make sure jupyterlab is updated
]

success_count = 0
for pkg in packages:
    if install(pkg):
        success_count += 1

print(f"\n{'='*50}")
print(f"Installed {success_count}/{len(packages)} packages successfully")

Installing torch==2.1.0...
  ✓ torch==2.1.0 installed successfully
Installing torchaudio==2.1.0...
  ✓ torchaudio==2.1.0 installed successfully
Installing torchvision==0.16.0...
  ✓ torchvision==0.16.0 installed successfully
Installing numpy==1.24.4...
  ✓ numpy==1.24.4 installed successfully
Installing scipy==1.11.4...
  ✓ scipy==1.11.4 installed successfully
Installing librosa==0.10.1...
  ✓ librosa==0.10.1 installed successfully
Installing soundfile==0.12.1...
  ✓ soundfile==0.12.1 installed successfully
Installing musdb==0.4.2...
  ✓ musdb==0.4.2 installed successfully
Installing stempeg==0.2.3...
  ✓ stempeg==0.2.3 installed successfully
Installing matplotlib==3.7.3...
  ✓ matplotlib==3.7.3 installed successfully
Installing tqdm==4.66.1...
  ✓ tqdm==4.66.1 installed successfully
Installing ipywidgets==8.1.1...
  ✓ ipywidgets==8.1.1 installed successfully
Installing jupyter-lab==4.0.8...
  ✗ ERROR installing jupyter-lab==4.0.8:
ERROR: Could not find a version that satisfies the req

In [ ]:
# Cell 4: Install PyTorch Geometric (needed for Phase 2 GCN)
# This is trickier - needs to match your PyTorch version

import torch
torch_version = torch.__version__.split('+')[0]  # e.g., "2.1.0"
cuda_version = "cu118"  # Change to cu121 if you have CUDA 12.1

print(f"Your PyTorch: {torch_version}")
print(f"Using CUDA version: {cuda_version}")
print("Installing PyTorch Geometric...")

# Install PyG and its dependencies
pyg_packages = [
    "torch-scatter",
    "torch-sparse", 
    "torch-geometric",
]

import subprocess, sys

# First install the wheels
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html"
print(f"Using wheel URL: {wheel_url}")

for pkg in pyg_packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, 
         "-f", wheel_url, "--quiet"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {pkg} installed")
    else:
        # Try without the wheel URL (might work with newer versions)
        result2 = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
            capture_output=True, text=True
        )
        if result2.returncode == 0:
            print(f"  ✓ {pkg} installed (fallback method)")
        else:
            print(f"  ✗ {pkg} FAILED - we'll handle this in Phase 2")

In [ ]:
# Cell 5: Create the entire project folder structure
import os

# Change this if you want the project somewhere else
PROJECT_ROOT = "D:/gsn_project"

# All folders to create
folders = [
    f"{PROJECT_ROOT}",
    f"{PROJECT_ROOT}/src",
    f"{PROJECT_ROOT}/src/audio",
    f"{PROJECT_ROOT}/src/data", 
    f"{PROJECT_ROOT}/src/models",
    f"{PROJECT_ROOT}/src/training",
    f"{PROJECT_ROOT}/src/utils",
    f"{PROJECT_ROOT}/notebooks",
    f"{PROJECT_ROOT}/configs",
    f"{PROJECT_ROOT}/checkpoints",
    f"{PROJECT_ROOT}/outputs",
    f"{PROJECT_ROOT}/outputs/audio",
    f"{PROJECT_ROOT}/outputs/plots",
    f"{PROJECT_ROOT}/logs",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"  ✓ Created: {folder}")

print(f"\nProject root: {PROJECT_ROOT}")
print("Folder structure created successfully!")